# Day 5 Runner — Honest Evaluation + Engineering

The honest table (base/CPT/SFT/DPO/instruct), a position-swapped win rate with
a Wilson CI, the CPT-usefulness ablation, a failure-analysis draft, and an
auto-generated reproducibility doc.

Settings in `configs/day5.yaml`.

## 0. GPU + code + install

In [ ]:
!nvidia-smi

In [ ]:
#!unzip llm-from-base-to-assistant.zip
%cd llm-from-base-to-assistant
import os; print(os.getcwd())

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

# Set BEFORE importing transformers, vllm, or huggingface_hub.
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
os.environ["HF_HOME"] = "/workspace/.cache/huggingface"
os.environ["HF_HUB_CACHE"] = "/workspace/.cache/huggingface/hub"
os.environ["HF_DATASETS_CACHE"] = "/workspace/.cache/huggingface/datasets"

REPO = Path("/workspace/llm-from-base-to-assistant")
if not (REPO / "configs/day4.yaml").is_file():
    raise FileNotFoundError(f"Place your repository at {REPO}, or edit REPO above.")
os.chdir(REPO)
print("Python:", sys.executable)
print("Repository:", REPO)

In [ ]:
# Run once. After it finishes, RESTART THE KERNEL, then go to 'After restart'.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "outlines"], check=False)

# 1a. Pins first, alone — no requirements.txt in this transaction.
inference_stack = [
    "vllm==0.10.2",
    "torch==2.8.0+cu128", "torchvision==0.23.0+cu128", "torchaudio==2.8.0+cu128",
    "transformers==4.56.1",
    "pyyaml", "datasets", "huggingface_hub",
]
subprocess.run([
    sys.executable, "-m", "pip", "install", "--upgrade", *inference_stack,
    "--extra-index-url", "https://download.pytorch.org/whl/cu128",
], check=True)

# 1b. Repo's OTHER requirements, if any — separate transaction so a stale pin
#     here can't abort the stack above. If pip reports a conflict, fix that
#     line in requirements.txt (don't force an unconstrained reinstall).
req = REPO / "requirements.txt"
if req.exists():
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(req),"--ignore-installed blinker"])
    if r.returncode != 0:
        print("requirements.txt conflicts with the pinned stack — edit the offending pin, then rerun THIS sub-step only.")

subprocess.run([sys.executable, "-m", "pip", "check"], check=False)
print("SETUP FINISHED. Restart the kernel, then start at 'After restart'.")

In [ ]:
subprocess.run([
    sys.executable, "-m", "pip", "install", "--upgrade",
    "trl>=0.23,<0.24", "peft>=0.17,<0.18", "pytest",
], check=True)
subprocess.run([sys.executable, "-m", "pip", "check"], check=False)

from importlib.metadata import version
for name in ("trl", "peft"):
    print(name, version(name))
# Sanity: transformers/torch unchanged by the above.
for name in ("torch", "vllm", "transformers"):
    print(name, version(name))

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
os.environ["HF_HOME"] = "/workspace/.cache/huggingface"
os.environ["HF_HUB_CACHE"] = "/workspace/.cache/huggingface/hub"
os.environ["HF_DATASETS_CACHE"] = "/workspace/.cache/huggingface/datasets"

REPO = Path("/workspace/llm-from-base-to-assistant")
if not (REPO / "configs/day3.yaml").is_file():
    raise FileNotFoundError(f"Place your repository at {REPO}, or edit REPO above.")
os.chdir(REPO)

import json
import shutil
import yaml
import torch
import vllm
from importlib.metadata import version, PackageNotFoundError
from vllm.model_executor.models import ModelRegistry

for name in ("torch", "vllm", "transformers"):
    print(name, version(name))
assert version("vllm") == "0.10.2"
assert version("transformers") == "4.56.1"
assert torch.cuda.is_available(), "CUDA unavailable: check host driver and installed wheel."
print("CUDA runtime:", torch.version.cuda, "| GPU:", torch.cuda.get_device_name(0))
assert "Qwen3ForCausalLM" in ModelRegistry.get_supported_archs()

In [ ]:
from huggingface_hub import login

login()

## 1. Quick tests (no GPU needed)

In [ ]:
import sys
!{sys.executable} -m pytest tests/test_day5_eval.py -q

## Leakage: hopefully CLEAN. If it says LEAK, put that note next to the domain-perplexity numbers.

In [ ]:
import sys
!{sys.executable} evaluation/leakage_check.py --config configs/day5.yaml

## 2. The honest table: base, CPT, SFT, DPO, instruct
Perplexity (domain + general, on the LOCKED held-out sets) + behavior (does it
answer, does it stop) for all five models. **Expect:** instruct wins general
breadth; your pipeline is competitive/wins on domain.

In [ ]:
import sys
!{sys.executable} evaluation/eval_suite.py --config configs/day5.yaml

## 3. Win rate: dpo-v1 vs instruct (proper position-swap)
Each pair judged TWICE (both slot orders); disagreement = tie, not a forced
winner. Reports the win rate **with sample count** and a 95% Wilson CI.

In [ ]:
import sys
!{sys.executable} evaluation/win_rate_vs_instruct.py --config configs/day5.yaml

In [ ]:
   import json
   r = json.load(open("evaluation/day5_win_rate.json"))
   for row in r["rows"]:
       if row["result"] == "instruct":
           print(f"[{row['group']}] dpo={row['dpo']['tokens']} tok, instruct={row['instruct']['tokens']} tok")
           print("   reason:", row["round1_A_is_dpo"]["verdict"]["reason"])

## 4. Ablation: was CPT worth it?
Trains SFT from the ORIGINAL base (skip CPT) using the exact same data and
hyperparameters as your real Day 3 run -- only one variable changes.

In [ ]:
import sys
!{sys.executable} training/sft_from_base.py --config configs/day5.yaml

In [ ]:
# add the ablation checkpoint to the models list, then re-run the honest table
# to compare it against sft-v1 (does CPT show up as a measurable difference?)
import yaml
cfg = yaml.safe_load(open("configs/day5.yaml"))
cfg["models"]["sft_from_base"] = "artifacts/sft-from-base-v1"
yaml.safe_dump(cfg, open("configs/day5.yaml", "w"))
print("Added sft_from_base to configs/day5.yaml -- re-run eval_suite.py to compare.")

In [ ]:
import sys
!{sys.executable} evaluation/eval_suite.py --config configs/day5.yaml

## 5. Failure analysis (draft)
Collects the worst-scoring sft-v1/dpo-v1 answers into a markdown draft. **You
fill in the WHY and WHAT-NEXT by hand** -- that reasoning is the real work.

In [ ]:
import sys
!{sys.executable} evaluation/failure_analysis.py --config configs/day5.yaml

In [ ]:
print(open("docs/failure_analysis_draft.md").read())

## 6. Reproducibility doc (no GPU needed)
Reads every stage's lineage.json and writes the literal rebuild steps. Flags
any missing field as a real gap to go fix.

In [ ]:
import sys
!{sys.executable} evaluation/build_reproduce_doc.py --config configs/day5.yaml

In [ ]:
print(open("docs/REPRODUCE.md").read())